## <<- Load the data ->>

In [45]:
import pandas as pd

# Load the main data file that will be split
epl_data = pd.read_csv('Data_Files/epl-training.csv')

# Calculate the split index for an 80/20 split
split_index = int(len(epl_data) * 0.8)

# Split the data into training and validation sets
epl_training = epl_data[:split_index].copy()
epl_validation = epl_data[split_index:].copy()

# Load the separate test file for final predictions
epl_test = pd.read_csv('Data_Files/epl-test.csv')

print(f"Total original data size: {len(epl_data)}")
print(f"Training set size (80%): {len(epl_training)}")
print(f"Validation set size (20%): {len(epl_validation)}")
print(f"Final test set size: {len(epl_test)}")

Total original data size: 9600
Training set size (80%): 7680
Validation set size (20%): 1920
Final test set size: 10


In [46]:
# Display the first few rows of the new training data to verify
epl_training.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR
0,19/08/2000,Charlton,Man City,4,0,H,2,0,H,Rob Harris,...,14,4,6,6,13,12,1,2,0,0
1,19/08/2000,Chelsea,West Ham,4,2,H,1,0,H,Graham Barber,...,10,5,7,7,19,14,1,2,0,0
2,19/08/2000,Coventry,Middlesbrough,1,3,A,1,1,D,Barry Knight,...,3,9,8,4,15,21,5,3,1,0
3,19/08/2000,Derby,Southampton,2,2,D,1,2,A,Andy D'Urso,...,4,6,5,8,11,13,1,1,0,0
4,19/08/2000,Leeds,Everton,2,0,H,2,0,H,Dermot Gallagher,...,8,6,6,4,21,20,1,3,0,0


In [47]:
# Display the first few rows of the new validation data to verify
epl_validation.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR
7680,21/11/2020,Tottenham,Man City,2,0,H,1,0,H,M Dean,...,2,5,0,10,13,19,2,2,0,0
7681,21/11/2020,Man United,West Brom,1,0,H,0,0,D,D Coote,...,7,2,8,2,12,17,1,1,0,0
7682,22/11/2020,Fulham,Everton,2,3,A,1,3,A,A Madley,...,6,7,2,5,14,10,2,0,0,0
7683,22/11/2020,Sheffield United,West Ham,0,1,A,0,0,D,M Atkinson,...,5,3,2,6,7,6,0,0,0,0
7684,22/11/2020,Leeds,Arsenal,0,0,D,0,0,D,A Taylor,...,4,2,5,3,9,8,3,0,0,1


In [48]:
# Display the first few rows of the new testing data to verify
epl_test.head()

,Date,HomeTeam,AwayTeam
0,31 Jan 26,Leeds,Arsenal
1,31 Jan 26,Liverpool,Newcastle
2,31 Jan 26,Tottenham,Man City
3,31 Jan 26,Wolves,Bournemouth
4,31 Jan 26,Aston Villa,Brentford


In [49]:

def prepare_epl_data(df):
    """
    Prepares the EPL data in the same double-entry format as the NCAA notebook.
    T1 is the home team, T2 is the away team. Then they are swapped.
    """
    # Select relevant columns and create GoalDiff
    df_prep = df[['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG']].copy()
    df_prep['GoalDiff'] = df_prep['FTHG'] - df_prep['FTAG']
    
    # Define the season based on the date
    df_prep['Date'] = pd.to_datetime(df_prep['Date'], dayfirst=True)
    df_prep['Season'] = df_prep['Date'].apply(lambda x: f"{x.year-1}-{x.year}" if x.month < 8 else f"{x.year}-{x.year+1}")

    # Rename for T1 and T2
    df1 = df_prep.rename(columns={'HomeTeam': 'T1_Team', 'AwayTeam': 'T2_Team'})
    
    # Create the swapped version
    df2 = df_prep.rename(columns={'HomeTeam': 'T2_Team', 'AwayTeam': 'T1_Team'})
    df2['GoalDiff'] = -df2['GoalDiff'] # Invert goal difference for the swapped perspective
    
    # Concatenate and return
    return pd.concat([df1, df2]).reset_index(drop=True)

In [50]:
import statsmodels.api as sm
import tqdm

def calculate_team_quality(data):
    """
    Calculates team 'quality' scores for each season using a GLM,
    similar to the 'Hardest difficulty features' in the NCAA notebook.
    """
    glm_quality_list = []
    seasons = sorted(data['Season'].unique())
    
    print("Calculating team quality for each season...")
    for season in tqdm.tqdm(seasons, unit="season"):
        # Filter data for the current season
        season_data = data[data['Season'] == season].copy()
        
        # Define the model formula
        formula = "GoalDiff ~ -1 + T1_Team + T2_Team"
        
        # Fit the GLM model
        glm = sm.GLM.from_formula(
            formula=formula,
            data=season_data,
            family=sm.families.Gaussian(),
        ).fit()
        
        # Extract parameters (the quality scores)
        quality = pd.DataFrame(glm.params).reset_index()
        quality.columns = ["Team", "quality"]
        quality["Season"] = season
        
        # Clean up team names from the formula format
        quality['Team'] = quality['Team'].apply(lambda x: x.split('[T.')[1][:-1] if '[T.' in x else '')
        quality = quality[quality['Team'] != ''].reset_index(drop=True)
        
        glm_quality_list.append(quality)
        
    return pd.concat(glm_quality_list).reset_index(drop=True)

In [51]:
prepared_data = prepare_epl_data(epl_training)
team_quality = calculate_team_quality(prepared_data)

Calculating team quality for each season...


100%|██████████| 21/21 [00:00<00:00, 33.16season/s]


In [52]:
prepared_data.head()

,Date,T1_Team,T2_Team,FTHG,FTAG,GoalDiff,Season
0,2000-08-19,Charlton,Man City,4,0,4,2000-2001
1,2000-08-19,Chelsea,West Ham,4,2,2,2000-2001
2,2000-08-19,Coventry,Middlesbrough,1,3,-2,2000-2001
3,2000-08-19,Derby,Southampton,2,2,0,2000-2001
4,2000-08-19,Leeds,Everton,2,0,2,2000-2001


In [53]:
team_quality.head()

,Team,quality,Season
0,Aston Villa,0.550,2000-2001
1,Bradford,1.625,2000-2001
2,Charlton,0.800,2000-2001
3,Chelsea,0.050,2000-2001
4,Coventry,1.300,2000-2001
